# Session 5: Exercises

**Do these before session 6.** About ninety minutes; the ⭐⭐⭐ ones join the session's ideas together.

Same pattern as before: **predict, then measure.** Write down what you expect before you run a cell.
Predictions and answers go in the cells marked **✍️**; code to write is marked `# TODO`. Each
exercise has a check cell that says `passed` or tells you what's wrong.

Exercises 1, 2 and 6's first half need no key. The rest call Gemini on the free tier.

Worked solutions are released after the next session.

## Setup (just run it)

In [ ]:
import json
import logging
import sys
from pathlib import Path

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain.tools import ToolRuntime, tool
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel

sys.path.insert(0, str(Path.cwd() / "project"))
from config import settings                                      # noqa: E402
import tools as shop                                             # noqa: E402
from agent import MAX_STEPS, answer, transcript                  # noqa: E402
from shopwise_data import CUSTOMERS, ORDERS, PRODUCTS, SHIPMENTS  # noqa: E402
from triage import PROMPTS                                       # noqa: E402
import score_tools                                               # noqa: E402

logging.getLogger("google_genai.types").setLevel(logging.ERROR)   # lesson setup explains this
model = init_chat_model(settings.model_id)


class ScriptedModel(GenericFakeChatModel):
    """Replays a fixed list of messages, so a loop can be tested with no key (lesson appendix E)."""

    def bind_tools(self, tools, **kwargs):
        return self


print(settings.model_id, "·", [t.name for t in shop.TOOLS])

## ⭐ Exercise 1: What the model sees

**The idea.** Lesson section 3. The model never sees your function, only its schema. Here's a tool
somebody wrote for listing orders with filters:

In [ ]:
@tool
def search_orders(customer_id: str, runtime: ToolRuntime, status: str = "any", limit: int = 5) -> list[dict]:
    """Search a customer's orders, newest first, optionally by status."""
    found = [{"order_id": oid, **o} for oid, o in ORDERS.items() if o["customer_id"] == customer_id]
    if status != "any":
        found = [o for o in found if o["status"] == status]
    return found[:limit]

**✍️ Predict first:** which arguments will the model be shown, and which must it fill in?

**✍️ Your prediction:**

*(write it here)*

Write `model_arguments(t)`: given a tool, return a dict mapping each argument **the model is shown**
to `True` if it's required and `False` if it's optional. Use `t.tool_call_schema.model_json_schema()`,
which is the schema the model receives.

In [ ]:
def model_arguments(t) -> dict[str, bool]:
    # TODO: read t.tool_call_schema.model_json_schema(); "properties" lists what the model sees,
    #       "required" lists what it must fill in.
    return {}

In [ ]:
def check_1():
    cases = [
        (shop.get_order, {"order_id": True}),
        (shop.issue_refund, {"order_id": True, "amount": True, "reason": True}),
        (search_orders, {"customer_id": True, "status": False, "limit": False}),
    ]
    for t, expected in cases:
        got = model_arguments(t)
        if got != expected:
            return f"{t.name}: expected {expected}, got {got}"
    return "passed"


print(check_1())

Now delete a type hint. Make a copy of `search_orders` with `limit` untyped (`limit=5`) and print
what the model would receive for it.

In [ ]:
# TODO: define search_orders_untyped, identical except `limit=5` has no type hint,
#       then print search_orders_untyped.tool_call_schema.model_json_schema()["properties"]["limit"]

**✍️** What does the model know about `limit` without the hint? And `status` accepts any string: how
would you tell the model which statuses exist, so it can't send `"in transit"` when the data says
`"shipped"`? (Hint: `typing.Literal`.)

**✍️ Your answer:**

*(write it here)*

## ⭐ Exercise 2: Audit a transcript

**The idea.** Lesson sections 4.5 and 4.6. Here is a saved run, as the message list `create_agent`
returned it. No key needed.

In [ ]:
saved = [
    HumanMessage("I'm tanvir@example.com. What's the status of ORD-5002 and ORD-5003?"),
    AIMessage("", tool_calls=[{"name": "lookup_customer", "args": {"email": "tanvir@example.com"}, "id": "a", "type": "tool_call"}]),
    ToolMessage('{"customer_id": "C-1002", "name": "Tanvir Ahmed"}', tool_call_id="a", name="lookup_customer"),
    AIMessage("", tool_calls=[{"name": "get_order", "args": {"order_id": "ORD-5002"}, "id": "b", "type": "tool_call"},
                              {"name": "get_order", "args": {"order_id": "ORD-5003"}, "id": "c", "type": "tool_call"}]),
    ToolMessage('{"order_id": "ORD-5003", "status": "shipped"}', tool_call_id="c", name="get_order"),
    ToolMessage('{"order_id": "ORD-5002", "status": "processing"}', tool_call_id="b", name="get_order"),
    AIMessage("ORD-5002 is still processing and ORD-5003 has shipped."),
]

**✍️ Predict first:** how many model calls did this run make, and how many tool executions? Is the
message list valid for Anthropic or OpenAI, given that the two results came back in the opposite
order to the requests?

**✍️ Your prediction:**

*(write it here)*

Write two functions:

- `count(messages)` returns `(model_calls, tool_runs)`.
- `unanswered(messages)` returns the IDs of tool requests with no `ToolMessage` answering them, in
  the order they were requested. This is the check a provider runs before rejecting your list.

In [ ]:
def count(messages):
    # TODO: an AIMessage is one model call; a ToolMessage is one tool run.
    return 0, 0


def unanswered(messages):
    # TODO: collect every tool_call id from AI messages, then remove each ToolMessage's tool_call_id.
    return []

In [ ]:
def check_2():
    if count(saved) != (3, 3):
        return f"count(saved): expected (3, 3), got {count(saved)}"
    if unanswered(saved) != []:
        return f"unanswered(saved): expected [], got {unanswered(saved)}"
    missing_one = saved[:5]                       # the result for request "b" never arrives
    if unanswered(missing_one) != ["b"]:
        return f"unanswered with one result missing: expected ['b'], got {unanswered(missing_one)}"
    if unanswered(saved[:2]) != ["a"]:
        return f"unanswered(saved[:2]): expected ['a'], got {unanswered(saved[:2])}"
    return "passed"


print(check_2())

**✍️** Where in a hand-written loop would you call `unanswered()`, and what should happen if it
returns something?

**✍️ Your answer:**

*(write it here)*

## ⭐⭐ Exercise 3: A new tool from scratch

**The idea.** Lesson sections 3 and 7. Customers ask "is it available?" far more often than they ask
about warranties. Write `check_stock(sku_or_name)` against the lesson's checklist:

- a verb name and a typed argument with an `Args:` line (`parse_docstring=True`);
- a first docstring line saying what it does **and what comes back**;
- returns `{"sku", "name", "in_stock", "stock"}` for one match;
- for no match or several matches, an `{"error": ...}` that says what would work, with the
  candidates or the available products, like `get_product` does.

**✍️ Predict first:** the kettle (`SKU-300`) has `stock: 0`. What should `check_stock("kettle")`
return, and why would `{"error": "out of stock"}` be a bug?

**✍️ Your prediction:**

*(write it here)*

In [ ]:
# TODO: write check_stock. Reuse the matching rule from shop.get_product (a SKU, or a
#       case-insensitive substring of the name). PRODUCTS is already imported.

In [ ]:
def check_3():
    kettle = check_stock.invoke({"sku_or_name": "kettle"})
    if kettle.get("in_stock") is not False or kettle.get("stock") != 0 or "error" in kettle:
        return f"the kettle is a successful lookup with in_stock False: got {kettle}"
    watch = check_stock.invoke({"sku_or_name": "SKU-200"})
    if watch.get("in_stock") is not True:
        return f"SKU-200 is in stock: got {watch}"
    missing = check_stock.invoke({"sku_or_name": "toaster"})
    if "error" not in missing or "available" not in missing:
        return f"no match should be an error listing what exists: got {missing}"
    several = check_stock.invoke({"sku_or_name": "p"})            # matches several names
    if "error" not in several or "candidates" not in several:
        return f"several matches should be an error listing candidates: got {several}"
    if "Returns:" in (check_stock.description or ""):
        return "say what comes back in the summary line: a Returns: section is dropped"
    return "passed"


print(check_3())

Give it to the agent alongside the six, and ask:

In [ ]:
with_stock = create_agent(model, shop.TOOLS + [check_stock], system_prompt=PROMPTS["agent_prompt"])
for q in ["Is the kettle available?", "Do you have the PulseFit smartwatch in stock?"]:
    print(transcript(answer(with_stock, q)), "\n")

Now the question that matters: did adding a seventh tool that overlaps `get_product` change tool
selection on the product questions? Run them with the extra tool:

In [ ]:
s = score_tools.score(with_stock, score_tools.load_questions("product"))
print(f"product questions with check_stock added: {s['passed']}/{s['total']}")

**✍️** Two questions. (1) The score counts `get_product` as the must-call tool. If the agent used
`check_stock` instead, is that a failure of the agent or of the test? (2) Should ShopWise have both
tools? Argue one side in three sentences.

**✍️ Your answer:**

*(write it here)*

## ⭐⭐ Exercise 4: Make a bad tool fail well

**The idea.** Lesson section 8. A colleague's version of `list_orders`. It passes the obvious test
(*"does Tanvir get his order back?"*) and has three real problems.

In [ ]:
@tool
def list_orders_v0(customer_id: str) -> list[dict]:
    """List a customer's orders."""
    known_ids = [c["customer_id"] for c in CUSTOMERS.values()]
    if customer_id not in known_ids:
        raise KeyError(customer_id)
    orders = [o for o in ORDERS.values() if o["customer_id"] == customer_id]
    if not orders:
        return {"error": "customer has no orders"}
    return orders


print(list_orders_v0.invoke({"customer_id": "C-1002"}))       # the happy path
try:
    list_orders_v0.invoke({"customer_id": "c1002"})
except KeyError as err:
    print("c1002 -> KeyError", err)

**✍️ Predict first**, for each call, what the **agent** would end up telling the customer:
`list_orders_v0("C-1002")`, `list_orders_v0("c1002")` (the model's own typo), and
`list_orders_v0("C-1004")` (Kamal, who has never ordered).

**✍️ Your prediction:**

*(write it here)*

In [ ]:
broken = create_agent(model, [shop.lookup_customer, list_orders_v0, shop.get_order],
                      system_prompt=PROMPTS["agent_prompt"])
for q in ["I'm tanvir@example.com. List my orders with their order numbers.",
          "I'm kamal@example.com. Have I ordered anything?"]:
    try:
        print(transcript(answer(broken, q)), "\n")
    except Exception as err:
        print("RUN FAILED:", type(err).__name__, err, "\n")

Rewrite it as `list_orders_v1`, keeping the same signature. Decide for each problem whether to
**fix** it, **return** an error, or **raise**.

In [ ]:
# TODO: write list_orders_v1 with @tool(parse_docstring=True).

In [ ]:
def check_4():
    tanvir = list_orders_v1.invoke({"customer_id": "C-1002"})
    if not isinstance(tanvir, list) or tanvir[0].get("order_id") != "ORD-5003":
        return f"Tanvir's orders must include order_id: got {tanvir}"
    kamal = list_orders_v1.invoke({"customer_id": "C-1004"})
    if kamal != []:
        return f"no orders is an empty list, not an error: got {kamal}"
    try:
        unknown = list_orders_v1.invoke({"customer_id": "C-9999"})
    except Exception as err:
        return f"an unknown ID is something the model can fix: return an error, don't raise {type(err).__name__}"
    if "error" not in unknown:
        return f"an unknown ID should come back as {{'error': ...}}: got {unknown}"
    return "passed"


print(check_4())

**✍️** Write your four decisions in one line each: the unknown ID, the missing order IDs, the empty
result, and the return type `list[dict] | dict`. Is that last one acceptable, and what would the
tidier alternatives cost?

**✍️ Your answer:**

*(write it here)*

## ⭐⭐ Exercise 5: Four ways to say "no tracking"

**The idea.** Lesson section 8.3: an error message is a prompt. `ORD-5002` exists, but it hasn't
shipped, so `get_shipment` has nothing to return. Four versions of that one error. The third was
copied from another tool's error handler by someone in a hurry.

In [ ]:
ERRORS = {
    "bare": "not found",
    "shipped as v2": "no tracking for ORD-5002. get_order shows its current status.",
    "copied": "No shipment found for that order ID. Ask the customer to double-check the order number.",
    "yours": "TODO: write your own",
}

**✍️ Predict first:** which version, if any, will make the agent tell the customer something false?

**✍️ Your prediction:**

*(write it here)*

Replace `"yours"` with your best wording, then run each version three times and read the replies.

In [ ]:
others = [t for t in shop.TOOLS if t.name != "get_shipment"]

for label, message in ERRORS.items():
    # A new get_shipment each time round, returning this version's message.
    @tool("get_shipment")
    def get_shipment(order_id: str) -> dict:
        """Track a shipped order: carrier, tracking ID, status and estimated arrival date."""
        return {"error": message}

    variant = create_agent(model, others + [get_shipment], system_prompt=PROMPTS["agent_prompt"])
    print(f"--- {label}: {message}")
    for run in range(3):
        result = answer(variant, "Where is my parcel for ORD-5002?")
        print(f"  {score_tools.tools_called(result)}  {result['messages'][-1].text[:120]}")

**✍️** Which version produced a reply that is **wrong**, not just unhelpful? Why did the agent go
along with it? And what does that tell you about any text that comes back from a tool?

**✍️ Your answer:**

*(write it here)*

## ⭐⭐⭐ Exercise 6: The loop by hand, all the way

**The idea.** Lesson section 4. Write the whole loop as a function, correct on every vendor:

- append the `AIMessage` **as received** (thought signatures);
- answer **every** request in a message, not just the first;
- stop after `max_calls` model calls, whatever the model says;
- return the message list and a list of input-token counts, one per model call.

It must work with any model that has `.invoke` and returns `AIMessage`s, which is what makes it
testable with `ScriptedModel` and no key.

In [ ]:
def run_loop(model_with_tools, tools, question, max_calls=6):
    """Returns (messages, input_tokens_per_call)."""
    tools_by_name = {t.name: t for t in tools}
    messages = [SystemMessage(PROMPTS["agent_prompt"]), HumanMessage(question)]
    tokens = []
    # TODO
    return messages, tokens

In [ ]:
def check_6():
    def call(name, args, call_id):
        return {"name": name, "args": args, "id": call_id, "type": "tool_call"}

    # Two requests in one message, then an answer.
    parallel = ScriptedModel(messages=iter([
        AIMessage("", tool_calls=[call("get_order", {"order_id": "ORD-5002"}, "x"),
                                  call("get_order", {"order_id": "ORD-5003"}, "y")]),
        AIMessage("both looked up"),
    ]))
    messages, tokens = run_loop(parallel, shop.TOOLS, "status of two orders?")
    answered = {m.tool_call_id for m in messages if m.type == "tool"}
    if answered != {"x", "y"}:
        return f"every request needs a ToolMessage: answered {answered or 'none'}"
    if messages[-1].text != "both looked up" or len(tokens) != 2:
        return "the loop should stop at the first reply with no tool calls, after 2 model calls"

    # A model that never stops asking.
    endless = ScriptedModel(messages=iter(
        [AIMessage("", tool_calls=[call("get_product", {"sku_or_name": "kettle"}, f"k{i}")]) for i in range(20)]))
    messages, tokens = run_loop(endless, shop.TOOLS, "anything", max_calls=4)
    if len(tokens) != 4:
        return f"max_calls=4 must stop after 4 model calls, made {len(tokens)}"
    return "passed"


print(check_6())

Now on the real model: the parallel question, and the four-hop one.

In [ ]:
model_with_tools = model.bind_tools(shop.TOOLS)
for q in ["What's the status of ORD-5002, and of ORD-5003?",
          "Hi, I'm tanvir@example.com. What did I order most recently, and when will it arrive?"]:
    messages, tokens = run_loop(model_with_tools, shop.TOOLS, q)
    requests = sum(len(m.tool_calls) for m in messages if m.type == "ai")
    print(f"{len(tokens)} model calls, {requests} tool requests, input tokens {tokens}")
    print("  ", messages[-1].text[:120])

One more measurement: delete the system prompt (start the list with the `HumanMessage` only) and run
the four-hop question again.

In [ ]:
# TODO: run the four-hop question with no SystemMessage and compare route and answer.

**✍️** (1) Why are there always more model calls than lookups? (2) Why does each call's input-token
count grow? (3) What did removing the system prompt change, and what didn't it? (4) Which
`create_agent` argument would you miss first if you shipped your `run_loop`, and why?

**✍️ Your answer:**

*(write it here)*

## ⭐⭐⭐ Exercise 7: Measure a tool description

**The idea.** Lesson section 7. `get_order` and `get_shipment` overlap: both know an order's status.
The fifteen questions in `tool_questions.yml` don't test that boundary much. Write questions that do,
then break and fix a docstring and measure each version.

In [ ]:
# TODO: write at least four questions whose correct tool is unambiguous: two that must call
#       get_order and NOT need tracking, two that must call get_shipment. Same shape as the YAML.
boundary = [
    # {"question": "...", "must": ["get_order"], "about": "boundary"},
]

Now a docstring someone might write in a hurry, which claims shipping as well:

In [ ]:
@tool("get_order", parse_docstring=True)
def get_order_vague(order_id: str) -> dict:
    """Get all order and shipping information for an order, including delivery details.

    Args:
        order_id: An order ID like "ORD-5001".
    """
    return shop.get_order.invoke({"order_id": order_id})


not_get_order = [t for t in shop.TOOLS if t.name != "get_order"]
for label, version in [("as shipped", shop.get_order), ("vague", get_order_vague)]:
    variant = create_agent(model, not_get_order + [version], system_prompt=PROMPTS["agent_prompt"])
    print(label)
    s = score_tools.score(variant, boundary)
    print(f"  -> {s['passed']}/{s['total']}, {s['model_calls']} model calls\n")

**✍️** Report both scores. If the vague version scored the same, does that mean the docstring doesn't
matter? What would you need to see before you trusted a change to a tool description in production?

**✍️ Your answer:**

*(write it here)*